# Notebook 3: 베이스라인 모델 학습
Input: Triage 코드 + Notebook1 Output (데이터) + Notebook2 Output (Mamba 체크포인트)

In [ ]:
# 1. GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU!')

In [ ]:
# 2. 코드 복사
import os, shutil

def find_input_dir():
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'configs' in dirs and 'src' in dirs:
            return root
    return None

input_dir = find_input_dir()
assert input_dir, 'Triage 데이터셋을 찾을 수 없습니다'
dest = '/kaggle/working/Triage'
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree(input_dir, dest)
print('코드 복사 완료')

In [ ]:
# 3. 처리된 데이터 + Mamba 체크포인트 복사
import os, shutil, zipfile, glob
from pathlib import Path

def extract_zip(zip_path, extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)

# Notebook1 output (데이터)
nb1_zip = '/kaggle/input/notebooks/parkyunhu1/notebook722c6297be/_output_.zip'
print('Notebook1 압축 해제 중...')
extract_zip(nb1_zip, '/kaggle/working/nb1_output')
print('완료')

candidates = glob.glob('/kaggle/working/nb1_output/**/processed', recursive=True)
assert candidates, 'processed 폴더를 찾을 수 없습니다'
data_src = str(Path(candidates[0]).parent)
print('Data source:', data_src)

data_dest = '/kaggle/working/Triage/data'
if os.path.exists(data_dest):
    shutil.rmtree(data_dest)
shutil.copytree(data_src, data_dest)
print('데이터 복사 완료')

for split in ['train', 'val', 'test']:
    n = len(list(Path(f'{data_dest}/processed/{split}').glob('*.pt')))
    print(f'{split}: {n} 파일')

# Notebook2 output (IMST-Mamba 체크포인트)
nb2_zip_candidates = glob.glob('/kaggle/input/notebooks/parkyunhu1/*/_output_.zip')
nb2_zip = [z for z in nb2_zip_candidates if 'notebook722c6297be' not in z]

if nb2_zip:
    print('\nNotebook2 압축 해제 중...')
    extract_zip(nb2_zip[0], '/kaggle/working/nb2_output')
    ckpt_candidates = glob.glob('/kaggle/working/nb2_output/**/seed_*', recursive=True)
    if ckpt_candidates:
        results_src = str(Path(ckpt_candidates[0]).parent)
        shutil.copytree(results_src, '/kaggle/working/Triage/results', dirs_exist_ok=True)
        print('Mamba 체크포인트 복사 완료')
    else:
        print('Mamba 체크포인트 없음')
        os.makedirs('/kaggle/working/Triage/results/logs', exist_ok=True)
else:
    print('Notebook2 output 없음 (Mamba 제외)')
    os.makedirs('/kaggle/working/Triage/results/logs', exist_ok=True)

In [ ]:
# 4. 패키지 설치
!pip install -q omegaconf lifelines statsmodels pyarrow

In [ ]:
# 5. Config 설정
import re, os
os.chdir('/kaggle/working/Triage')

with open('configs/base.yaml') as f:
    content = f.read()
content = re.sub(r'batch_size: \d+', 'batch_size: 64', content)
content = re.sub(r'max_epochs: \d+', 'max_epochs: 20', content)
content = re.sub(r'precision: \S+', 'precision: float16', content)
content = re.sub(r'early_stopping_patience: \d+', 'early_stopping_patience: 7', content)
with open('configs/base.yaml', 'w') as f:
    f.write(content)
print('Config 완료')

In [ ]:
# 6. 베이스라인 모델 학습
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'

for model in ['grud', 'lstm', 'transformer', 'retain']:
    print(f'\n=== {model} 학습 ===')
    result = subprocess.run(
        [sys.executable, 'scripts/run_pipeline.py', '--stage', 'train',
         '--model', model, '--seeds', '42'],
        env=env, capture_output=True, text=True,
        cwd='/kaggle/working/Triage'
    )
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])

In [ ]:
# 7. 전체 평가
import subprocess, sys, os, json

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Triage'

result = subprocess.run(
    [sys.executable, 'scripts/run_pipeline.py', '--stage', 'evaluate'],
    env=env, capture_output=True, text=True,
    cwd='/kaggle/working/Triage'
)
print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

# 결과 출력
try:
    with open('/kaggle/working/Triage/results/evaluation_results.json') as f:
        results = json.load(f)
    print('\n=== 최종 결과 ===')
    for model, r in results.items():
        print(f"{model}: AUROC={r['auroc']:.4f}, AUPRC={r['auprc']:.4f}")
except:
    pass

# Output에 전체 results 저장
import shutil
if os.path.exists('/kaggle/working/results'):
    shutil.rmtree('/kaggle/working/results')
shutil.copytree('/kaggle/working/Triage/results', '/kaggle/working/results')
print('Output 저장 완료')